# Output and Visualisation

JuPedSim records agent trajectories to an **SQLite file** via
`jps.SqliteTrajectoryWriter`.  The file stores positions at every simulation step
and can be replayed or analysed after the run.

> **Tip — `jps.Recording`**
> For in-memory recording (no disk I/O) use `jps.Recording`.
> It exposes the same trajectory data but keeps everything in RAM,
> which is useful for short interactive experiments.

This notebook shows how to:
1. Write trajectories to disk with `SqliteTrajectoryWriter`.
2. Read them back and animate the result.
3. Compute a speed metric with [PedPy](https://pedpy.readthedocs.io) and plot it.

See also: {doc}`Getting started </notebooks/fundamentals/00_getting_started>` ·
[JuPedSim scenarios](https://scenarios.jupedsim.org) for parameter sweeps.

In [ ]:
import pathlib

import jupedsim as jps
import shapely

trajectory_file = pathlib.Path("output.sqlite")

geometry = shapely.Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])

simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=trajectory_file, commit_every_nth_write=1
    ),
)

exit_id = simulation.add_exit_stage([(9, 4), (10, 4), (10, 6), (9, 6)])
journey_id = simulation.add_journey(jps.JourneyDescription([exit_id]))

n_agents = 20
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=exit_id,
            position=position,
            radius=0.12,
        )
    )

while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(
    f"Evacuated {n_agents} agents in {simulation.iteration_count()} iterations "
    f"({simulation.elapsed_time():.1f} s). Trajectories: {trajectory_file}"
)

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

In [ ]:
import matplotlib.pyplot as plt
import pedpy

speed = pedpy.compute_individual_speed(traj_data=trajectory_data, frame_step=5)
speed = speed.merge(trajectory_data.data, on=["id", "frame"], how="left")
plt.hist(speed["speed"], bins=30)
plt.xlabel("speed / m/s")
plt.ylabel("count")
plt.title("Distribution of individual speeds")
plt.show()

## Try one change

- Change `frame_step` (e.g. `frame_step=1` or `frame_step=10`) and observe how
  the speed estimate changes — larger steps smooth out instantaneous fluctuations.
- Plot speed over time instead of a histogram:

```python
mean_speed = speed.groupby("frame")["speed"].mean()
mean_speed.plot()
plt.xlabel("frame")
plt.ylabel("mean speed / m/s")
plt.show()
```

## See also

- {doc}`Getting started </notebooks/fundamentals/00_getting_started>` — first simulation from scratch.
- [JuPedSim scenarios](https://scenarios.jupedsim.org) for multi-scenario sweeps
  and advanced output analysis.